# When Does Graph Structure Help in MARL? --- Colab runner

This notebook reproduces every phase of the paper *When Does Graph Structure Help in Multi-Agent Reinforcement Learning?* in one place.

**What it does** (≈30--90 min wall on Colab CPU, ≈15--40 min on T4 GPU):

1. Clone the repo at the desired branch (default `phase-1-pilot`).
2. Install dependencies.
3. Run the unit-test suite as a smoke check.
4. Run each phase (1, 2, 4) sequentially. **Toggle phases in the *Run config* cell.**
5. Generate figures + summary tables under `results/<phase>/`.
6. Compile `paper/main.pdf` with the real results inserted.
7. Zip everything (results + paper PDF) into `gnnmarl_outputs.zip` and download it.

**Why use Colab?** The model is tiny (~30k params); GPU offers a modest 2--3x speedup over Colab CPU for this workload. The bigger reason is convenience and reproducibility.

**After the run**, unzip `gnnmarl_outputs.zip` into the repo's `results/` directory on your local machine; downstream analysis scripts and the paper build pick it up automatically.

## 1. Setup

Clone the repo and install the package. Change `BRANCH` if you want a different branch (e.g. `main` once Phase 1 is merged).

In [ ]:
REPO_URL = 'https://github.com/Evasion-OC/when-graphs-help-marl.git'
BRANCH   = 'phase-1-pilot-v2'   # change if needed

import os, subprocess, sys
if not os.path.exists('when-graphs-help-marl'):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL])
os.chdir('when-graphs-help-marl')
print('cwd:', os.getcwd())
subprocess.check_call(['git', 'log', '--oneline', '-1'])


In [ ]:
!pip install -q -e '.[dev]' 2>&1 | tail -5


## 1b. (Recommended) Mount Google Drive as a persistent backup

Colab runtimes disconnect after ~90 minutes of idle, or get pre-empted on the free tier. If that happens mid-experiment, everything in `/content/` is lost.

Mounting Drive lets us *checkpoint after each phase* into `MyDrive/gnnmarl-results/`. Even if the runtime dies between phases, the completed phases survive and you can rerun just the missing ones.

If you skip this cell, the notebook still works — you just have to babysit the tab to grab the download zip at the end.

In [ ]:
USE_DRIVE = True   # set False to skip
DRIVE_DIR = '/content/drive/MyDrive/gnnmarl-results'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Drive mounted; checkpoints will go to {DRIVE_DIR}')
else:
    DRIVE_DIR = None
    print('skipping Drive — outputs only available via the final zip download')


In [ ]:
# Helper: copy results + paper into Drive after each phase. Idempotent.
import shutil
from pathlib import Path

def checkpoint(label: str) -> None:
    if not DRIVE_DIR:
        return
    dst = Path(DRIVE_DIR) / label
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    for src in [Path('results'), Path('paper')]:
        if src.exists():
            shutil.copytree(src, dst / src.name, dirs_exist_ok=True)
    print(f'[checkpoint] {label} -> {dst}')


## 2. Smoke check

Run the full unit + integration test suite. Should report ~71 passed.

In [ ]:
!PYTHONIOENCODING=utf-8 python -m pytest -q 2>&1 | tail -5


## 3. GPU / device check

Colab gives you either a T4 GPU or CPU-only depending on the runtime type (Runtime > Change runtime type). The trainer auto-detects.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))


## 4. Run config

Toggle which phases to run. Defaults run all three reportable phases at their scaled budgets (matches what the paper reports).

If Colab disconnects mid-run, just re-run the affected phase cell; existing CSVs under `results/<phase>/<run_id>/episodes.csv` are not overwritten unless you clear that directory.

In [ ]:
RUN_PHASE_1 = True   # pilot: 4 algos x 3 seeds x 50k steps  (~30-90 min)
RUN_PHASE_2 = True   # E1 sweep: 4 algos x 2 N x 2 graphs x 3 seeds x 30k  (~50-120 min)
RUN_PHASE_4 = True   # depth x diameter ablation: 3 graphs x 4 depths x 3 seeds x 20k  (~30-90 min)


## 5. Phase 1 --- pilot

Trains each of {IQL, VDN, QMIX, GNN-QMIX} for 3 seeds x 50k env steps on CoordGrid ($N=4$, ring). Per-run wall is ~3--6 min depending on hardware.

In [ ]:
if RUN_PHASE_1:
    !PYTHONIOENCODING=utf-8 python scripts/phase1_pilot.py 2>&1 | tee results/phase1_log.txt
    !PYTHONIOENCODING=utf-8 python scripts/phase1_analysis.py 2>&1 | tail -20
    checkpoint('after_phase1')
else:
    print('skipped phase 1')


## 6. Phase 2 --- E1 sweep (H1, H2)

$N \in \{4, 8\} \times \{$ring, Erdős--Rényi (matched density)$\} \times$ 4 algos $\times$ 3 seeds at $3\times 10^4$ env steps per run. 48 runs total.

In [ ]:
if RUN_PHASE_2:
    !PYTHONIOENCODING=utf-8 python scripts/phase2_e1_sweep.py 2>&1 | tee results/phase2_log.txt
    !PYTHONIOENCODING=utf-8 python scripts/phase2_analysis.py 2>&1 | tail -30
    checkpoint('after_phase2')
else:
    print('skipped phase 2')


## 7. Phase 4 --- depth $\times$ diameter ablation (H3)

GNN depth $L \in \{1,2,3,4\}$ crossed with graph diameter $d \in \{1,2,3\}$ ($N=4$ fixed), 3 seeds, 20k env steps per run. 36 runs total.

In [ ]:
if RUN_PHASE_4:
    !PYTHONIOENCODING=utf-8 python scripts/phase4_ablation.py 2>&1 | tee results/phase4_log.txt
    !PYTHONIOENCODING=utf-8 python scripts/phase4_analysis.py 2>&1 | tail -20
    checkpoint('after_phase4')
else:
    print('skipped phase 4')


## 8. Build the paper

Regenerates the LaTeX inserts from the result CSVs and compiles `paper/main.pdf`. Requires `texlive` --- Colab usually has it but the install line is here just in case.

In [ ]:
import subprocess
try:
    subprocess.check_call(['which', 'pdflatex'])
    have_latex = True
except subprocess.CalledProcessError:
    have_latex = False
if not have_latex:
    !apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended texlive-science 2>&1 | tail -3


In [ ]:
!PYTHONIOENCODING=utf-8 python scripts/build_paper_inserts.py
%cd paper
!pdflatex -interaction=nonstopmode main.tex > /tmp/lp.txt 2>&1 ; bibtex main > /tmp/bt.txt 2>&1 ; pdflatex -interaction=nonstopmode main.tex > /tmp/lp.txt 2>&1 ; pdflatex -interaction=nonstopmode main.tex 2>&1 | tail -3
%cd ..


## 9. Bundle + download

Zips the `results/` directory and the compiled `paper/main.pdf` for download. After downloading, unzip into the repo's root on your local machine so the next iteration of analysis / writeup sees the data.

In [ ]:
import shutil, os, zipfile
from pathlib import Path

out = Path('gnnmarl_outputs.zip')
if out.exists(): out.unlink()

# Comprehensive bundle: results, figures, full paper source + PDF,
# review notes, and the reproducibility docs. Sized so a local unzip
# into the repo root restores a working tree.
files: list[Path] = []
for p in Path('results').rglob('*'):
    if p.is_file(): files.append(p)
for pattern in [
    'paper/main.tex',
    'paper/main.pdf',
    'paper/references.bib',
    'paper/REVIEW_NOTES.md',
    'paper/results/*.tex',
    'paper/figures/*',
    'docs/REPRO.md',
    'docs/PHASES.md',
    'docs/INTERFACES.md',
]:
    for p in Path('.').glob(pattern):
        if p.is_file(): files.append(p)

with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p)
print(f'wrote {out} -- {out.stat().st_size / 1e6:.1f} MB, {len(files)} files')

# Belt-and-suspenders: copy the zip to Drive if mounted, so it survives
# Colab disconnect even if you don't accept the browser download.
if DRIVE_DIR:
    drive_out = Path(DRIVE_DIR) / out.name
    shutil.copy2(out, drive_out)
    print(f'also copied to {drive_out}')
    # And a final phase-agnostic checkpoint of paper + results.
    checkpoint('final')

# Trigger the browser download. Requires you to accept the file dialog.
try:
    from google.colab import files as gcf
    gcf.download(str(out))
except Exception as e:
    print(f'download dialog skipped: {e}')
    print(f'grab the zip yourself from {out.resolve()}')


## 10. (Optional) Auto-push results to GitHub

Pushes the result CSVs, figures, and compiled paper to a fresh `results/colab-run-<utc-timestamp>` branch so the local working copy can pick them up with one command — no manual file transfer.

**Token setup** (in order of preference):

1. **Colab Secrets (recommended)** — in the left sidebar, click the key icon (*Secrets*), add a secret named `GITHUB_TOKEN`, paste a personal access token with `repo` scope, and toggle *Notebook access*. The cell picks it up automatically; it survives runtime restarts within the same Google account.
2. **Interactive prompt** — if no secret is set, you'll be prompted at runtime (input is masked).
3. **Skip** — set `PUSH_TO_GITHUB = False` below. The Drive checkpoint and the download zip are unaffected.

Create a PAT at https://github.com/settings/tokens (classic) with the `repo` scope, or use a fine-grained token scoped to this single repo with *Contents: write*.

In [ ]:
PUSH_TO_GITHUB = True   # set False to skip

if PUSH_TO_GITHUB:
    import subprocess, datetime, os, sys
    
    # 1. Resolve the token.
    token = ''
    try:
        from google.colab import userdata
        token = (userdata.get('GITHUB_TOKEN') or '').strip()
    except Exception:
        pass
    if not token:
        import getpass
        token = getpass.getpass('GitHub PAT (repo scope; input hidden): ').strip()
    
    if not token:
        print('No token provided -- skipping push.')
    else:
        ts = datetime.datetime.utcnow().strftime('%Y%m%d-%H%M%S')
        branch = f'results/colab-run-{ts}'
        
        # 2. Configure git identity (one-off, no global config touched).
        subprocess.run(['git', 'config', 'user.name',  'colab-runner'], check=True)
        subprocess.run(['git', 'config', 'user.email', 'colab-runner@users.noreply.github.com'], check=True)
        
        # 3. Create the new branch off whatever we cloned.
        subprocess.run(['git', 'checkout', '-b', branch], check=True)
        
        # 4. Stage outputs only (intentionally skip source changes).
        staged_any = False
        for path in ['results', 'paper/results', 'paper/figures', 'paper/main.pdf']:
            if os.path.exists(path):
                subprocess.run(['git', 'add', '-f', '--', path], check=True)
                staged_any = True
        if not staged_any:
            print('Nothing to commit -- did any phase actually run?')
        else:
            msg = f'Colab run {ts}: results + figures + compiled paper'
            r = subprocess.run(['git', 'commit', '-m', msg, '--allow-empty'],
                               capture_output=True, text=True)
            if r.returncode != 0 and 'nothing to commit' not in (r.stdout + r.stderr):
                print(f'commit issue:\n{r.stdout}\n{r.stderr}')
            
            # 5. Push via token-authenticated URL (single-use, never written
            #    to the remote config; we also scrub the token from any
            #    printed output before showing it).
            push_url = REPO_URL.replace('https://',
                                        f'https://x-access-token:{token}@')
            r = subprocess.run(['git', 'push', '-u', push_url, branch],
                               capture_output=True, text=True)
            
            def _scrub(s: str) -> str:
                return (s or '').replace(token, '<TOKEN>')
            
            if r.returncode == 0:
                view = REPO_URL.replace('.git', '') + f'/tree/{branch}'
                print(f'pushed: {branch}')
                print(f'view:   {view}')
                print()
                print('On your local machine, integrate with:')
                print(f'    git fetch origin')
                print(f'    git checkout origin/{branch} -- results/ paper/results/ paper/figures/ paper/main.pdf')
                print('then tell Claude the branch name and it will pick up the data.')
            else:
                print(f'push failed:\n{_scrub(r.stderr)}\n{_scrub(r.stdout)}')
        
        # Belt: clear the token from local memory once we're done with it.
        token = '<used-and-cleared>'


## What got saved, and where

**On GitHub** (if auto-push was enabled in cell 10):
- A new branch `results/colab-run-<timestamp>` containing the result CSVs, figures, and compiled paper PDF. The cell prints the URL.

**On Google Drive** (if mounted in cell 1b):
- `MyDrive/gnnmarl-results/after_phase1/` — full `results/` + `paper/` snapshot after Phase 1 finishes. Same for `after_phase2/`, `after_phase4/`, and `final/`. Each is a complete tree, so any one of them is independently usable.
- `MyDrive/gnnmarl-results/gnnmarl_outputs.zip` — the final bundle.

**On your local machine** (after accepting the download dialog):
- `gnnmarl_outputs.zip` in your browser's Downloads folder.

**Contents of the zip:**
- Every per-run `episodes.csv` + `config.json` under `results/phase{1,2,4}/`
- All learning-curve, heatmap, and forest-plot figures (PDF + PNG)
- Per-phase summary, pairwise, and hypothesis-test CSVs
- `paper/main.tex` (source), `paper/main.pdf` (compiled), `paper/references.bib`
- `paper/REVIEW_NOTES.md`, `paper/results/*.tex` (auto-generated inserts)
- `docs/{REPRO,PHASES,INTERFACES}.md`

## Local follow-up

```bash
cd path/to/when-graphs-help-marl
unzip ~/Downloads/gnnmarl_outputs.zip   # overlays results/ and paper/
open paper/main.pdf
```

If you edit the paper source and want to recompile against the same data:

```bash
bash scripts/build_paper.sh
```